# Chicago Food Inspections + 311 Sanitation — Final Data Cleaning

This notebook includes only the cleaning steps needed for the final datasets. Repeated exploration, tests, and intermediate versions have been removed.

## Cleaning Principles
1. **Do not fill missing values just to eliminate NaNs.** Impute only when other columns provide sufficient evidence.
2. **Preserve the original information.** Create cleaned/final columns for important variables before preparing the final dataset.
3. **Avoid unnecessary row deletion.** Remove only exact duplicates and a small number of rows missing essential analysis information that cannot be recovered.
4. **The main join keys are ZIP + Year.** Therefore, cleaning ZIP codes takes priority over fields not used in the main analysis.


## 1. Setup and Load Data

Both original datasets remain unchanged. All cleaning is performed on `food_test` and `sanitation_test`.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier, BallTree

food_path = "/Users/michael818/Desktop/BA780_dataCleaning/Food_Inspections_2010_2026.csv"
sanitation_path = "/Users/michael818/Desktop/BA780_dataCleaning/311_Sanitation_Combined_2010_2026.csv"

food = pd.read_csv(food_path)
sanitation = pd.read_csv(sanitation_path)

food_test = food.copy()
sanitation_test = sanitation.copy()

print("Food:", food_test.shape)
print("311:", sanitation_test.shape)

# Part A — Food Inspections

## 2. Facility Type

`Facility Type` has 5,339 missing values.  
We do not fill these using the overall mode because facility types differ across establishments.

Imputation priority:
1. **License #**: The closest available establishment identifier.
2. **DBA Name + Address**: Use historical records for the same name and address when the license does not provide a match.
3. **DBA Name**: Use only as a last resort because the same name may refer to different locations.

Previous holdout validation on known records showed approximate accuracy of:
- License #: **99.47%**
- DBA Name + Address: **99.42%**
- DBA Name: **98.28%**

For actual imputation, we also require **≥ 90% historical consistency**. `License # = 0` is not treated as a valid identifier.


In [ ]:
# Clean invalid license identifier
food_test["License_clean"] = food_test["License #"].replace(0, np.nan)

# Keep original values and record the source of every imputed value
food_test["Facility Type Imputed"] = food_test["Facility Type"]
food_test["Facility Type Imputation Method"] = np.where(
    food_test["Facility Type"].notna(),
    "Original",
    "Still Missing"
)

def mode_stats(df, group_cols, target):
    stats = (
        df.dropna(subset=group_cols + [target])
          .groupby(group_cols)[target]
          .agg(
              predicted=lambda x: x.mode().iloc[0],
              total_records="count",
              mode_count=lambda x: x.value_counts().iloc[0]
          )
    )
    stats["confidence"] = stats["mode_count"] / stats["total_records"]
    return stats

# ---------- 1) License # ----------
license_stats = mode_stats(
    food_test,
    ["License_clean"],
    "Facility Type"
)

license_prediction = food_test["License_clean"].map(license_stats["predicted"])
license_confidence = food_test["License_clean"].map(license_stats["confidence"])

mask = (
    food_test["Facility Type Imputed"].isna()
    & license_prediction.notna()
    & (license_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = license_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "License #"

# ---------- 2) DBA Name + Address ----------
dba_address_stats = mode_stats(
    food_test,
    ["DBA Name", "Address"],
    "Facility Type"
)

lookup_index = pd.MultiIndex.from_frame(food_test[["DBA Name", "Address"]])

dba_address_prediction = pd.Series(
    dba_address_stats["predicted"].reindex(lookup_index).to_numpy(),
    index=food_test.index
)
dba_address_confidence = pd.Series(
    dba_address_stats["confidence"].reindex(lookup_index).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Facility Type Imputed"].isna()
    & dba_address_prediction.notna()
    & (dba_address_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = dba_address_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "DBA Name + Address"

# ---------- 3) DBA Name ----------
dba_stats = mode_stats(
    food_test,
    ["DBA Name"],
    "Facility Type"
)

dba_prediction = food_test["DBA Name"].map(dba_stats["predicted"])
dba_confidence = food_test["DBA Name"].map(dba_stats["confidence"])

mask = (
    food_test["Facility Type Imputed"].isna()
    & dba_prediction.notna()
    & (dba_confidence >= 0.90)
)

food_test.loc[mask, "Facility Type Imputed"] = dba_prediction[mask]
food_test.loc[mask, "Facility Type Imputation Method"] = "DBA Name"

print("Facility Type imputation:")
print(food_test["Facility Type Imputation Method"].value_counts())
print("Remaining missing:", food_test["Facility Type Imputed"].isna().sum())

### Standardize Facility Type Labels

The original `Facility Type` contains many differences in capitalization, spacing, and punctuation, such as `GAS STATION` / `gas station` and `ROOF TOP` / `ROOFTOP`.

Approach:
- Create a comparison key using lowercase and trimmed whitespace.
- Keep the most common original spelling within each group as the canonical label.
- Apply further canonicalization to candidate groups that match after removing punctuation. These groups were previously reviewed manually to confirm that the differences reflect formatting or naming variations, rather than genuinely different facility types.


In [ ]:
# Basic formatting normalization
food_test["Facility Type Key"] = (
    food_test["Facility Type Imputed"]
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

canonical_basic = (
    food_test.dropna(subset=["Facility Type Imputed"])
    .groupby("Facility Type Key")["Facility Type Imputed"]
    .agg(lambda x: x.value_counts().index[0])
)

food_test["Facility Type Clean"] = (
    food_test["Facility Type Key"].map(canonical_basic)
)

# More aggressive comparison key for punctuation/spacing variants
food_test["Facility Compare"] = (
    food_test["Facility Type Clean"]
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

candidate_groups = (
    food_test.dropna(subset=["Facility Type Clean"])
    .groupby("Facility Compare")["Facility Type Clean"]
    .unique()
)

candidate_groups = candidate_groups[candidate_groups.apply(len) > 1]

# These groups were manually reviewed earlier and found to represent
# formatting/naming variants rather than substantively different categories.
final_mapping = {}

for _, values in candidate_groups.items():
    counts = (
        food_test.loc[
            food_test["Facility Type Clean"].isin(values),
            "Facility Type Clean"
        ]
        .value_counts()
    )
    canonical = counts.index[0]
    for value in values:
        final_mapping[value] = canonical

food_test["Facility Type Final"] = (
    food_test["Facility Type Clean"].replace(final_mapping)
)

print("Original categories:", food_test["Facility Type"].nunique())
print("Final categories:", food_test["Facility Type Final"].nunique())
print("Remaining missing:", food_test["Facility Type Final"].isna().sum())

## 3. Risk

`Risk` has only a small number of missing values.  
`All` is **not a missing value**: it mainly appears in special inspection contexts such as `License`, `Not Ready`, `No Entry`, and `Out of Business`, so it is preserved.

Missing Risk values are imputed only from high-confidence historical records, in this order:
1. License #
2. DBA Name + Address

Previous validation accuracy was approximately:
- License #: **99.71%**
- DBA Name + Address: **98.72%**

Values that cannot be reliably imputed remain NaN.


In [ ]:
normal_risk = [
    "Risk 1 (High)",
    "Risk 2 (Medium)",
    "Risk 3 (Low)"
]

food_test["Risk Imputed"] = food_test["Risk"]
food_test["Risk Imputation Method"] = np.where(
    food_test["Risk"].notna(),
    "Original",
    "Still Missing"
)

# ---------- 1) License # ----------
risk_license_stats = (
    food_test[
        food_test["Risk"].isin(normal_risk)
        & food_test["License_clean"].notna()
    ]
    .groupby("License_clean")["Risk"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
risk_license_stats["confidence"] = (
    risk_license_stats["mode_count"] / risk_license_stats["total_records"]
)

prediction = food_test["License_clean"].map(risk_license_stats["predicted"])
confidence = food_test["License_clean"].map(risk_license_stats["confidence"])

mask = (
    food_test["Risk Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Risk Imputed"] = prediction[mask]
food_test.loc[mask, "Risk Imputation Method"] = "License #"

# ---------- 2) DBA Name + Address ----------
risk_dba_address_stats = (
    food_test[food_test["Risk"].isin(normal_risk)]
    .dropna(subset=["DBA Name", "Address"])
    .groupby(["DBA Name", "Address"])["Risk"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
risk_dba_address_stats["confidence"] = (
    risk_dba_address_stats["mode_count"] /
    risk_dba_address_stats["total_records"]
)

idx = pd.MultiIndex.from_frame(food_test[["DBA Name", "Address"]])
prediction = pd.Series(
    risk_dba_address_stats["predicted"].reindex(idx).to_numpy(),
    index=food_test.index
)
confidence = pd.Series(
    risk_dba_address_stats["confidence"].reindex(idx).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Risk Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Risk Imputed"] = prediction[mask]
food_test.loc[mask, "Risk Imputation Method"] = "DBA Name + Address"

food_test["Risk Final"] = food_test["Risk Imputed"]

print(food_test["Risk Imputation Method"].value_counts())
print("Remaining missing:", food_test["Risk Final"].isna().sum())

## 4. ZIP

ZIP is a key field for merging with the 311 dataset, so recovering missing ZIP codes is a priority.

Imputation order:
1. **Exact Address match**
2. **Exact Latitude + Longitude match**
3. **Nearest coordinate within 100 meters**

Validation accuracy for the first two methods on known records was approximately **99.95%** and **99.94%**, respectively.  
The nearest-coordinate method accepts reference points only within 100 meters to avoid guessing ZIP codes from locations that are too far away.


In [ ]:
food_test["Zip Imputed"] = food_test["Zip"]
food_test["Zip Imputation Method"] = np.where(
    food_test["Zip"].notna(),
    "Original",
    "Still Missing"
)

# ---------- 1) Exact Address ----------
address_zip_stats = (
    food_test.dropna(subset=["Address", "Zip"])
    .groupby("Address")["Zip"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
address_zip_stats["confidence"] = (
    address_zip_stats["mode_count"] / address_zip_stats["total_records"]
)

prediction = food_test["Address"].map(address_zip_stats["predicted"])
confidence = food_test["Address"].map(address_zip_stats["confidence"])

mask = (
    food_test["Zip Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Zip Imputed"] = prediction[mask]
food_test.loc[mask, "Zip Imputation Method"] = "Address"

# ---------- 2) Exact coordinates ----------
coord_zip_stats = (
    food_test.dropna(subset=["Latitude", "Longitude", "Zip"])
    .groupby(["Latitude", "Longitude"])["Zip"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
coord_zip_stats["confidence"] = (
    coord_zip_stats["mode_count"] / coord_zip_stats["total_records"]
)

coord_index = pd.MultiIndex.from_frame(food_test[["Latitude", "Longitude"]])
prediction = pd.Series(
    coord_zip_stats["predicted"].reindex(coord_index).to_numpy(),
    index=food_test.index
)
confidence = pd.Series(
    coord_zip_stats["confidence"].reindex(coord_index).to_numpy(),
    index=food_test.index
)

mask = (
    food_test["Zip Imputed"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "Zip Imputed"] = prediction[mask]
food_test.loc[mask, "Zip Imputation Method"] = "Latitude + Longitude"

# ---------- 3) Nearest coordinate within 100m ----------
known_geo = food_test.dropna(
    subset=["Latitude", "Longitude", "Zip"]
).copy()

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(
    known_geo[["Latitude", "Longitude"]],
    known_geo["Zip"]
)

remaining_mask = (
    food_test["Zip Imputed"].isna()
    & food_test["Latitude"].notna()
    & food_test["Longitude"].notna()
)

if remaining_mask.any():
    X_missing = food_test.loc[
        remaining_mask,
        ["Latitude", "Longitude"]
    ]

    predicted_zip = knn.predict(X_missing)
    distances, _ = knn.kneighbors(X_missing)

    # Approximate conversion: 1 degree latitude ≈ 111 km
    approx_distance_km = distances[:, 0] * 111

    accepted_index = X_missing.index[approx_distance_km <= 0.1]

    predicted_series = pd.Series(
        predicted_zip,
        index=X_missing.index
    )

    food_test.loc[
        accepted_index,
        "Zip Imputed"
    ] = predicted_series.loc[accepted_index]

    food_test.loc[
        accepted_index,
        "Zip Imputation Method"
    ] = "Nearest Coordinate"

print(food_test["Zip Imputation Method"].value_counts())
print("Remaining missing ZIP:", food_test["Zip Imputed"].isna().sum())

## 5. City

The original `City` values include capitalization differences, repeated text, and obvious typos, such as:
`Chicago`, `chicago`, `CCHICAGO`, `CHICAGOO`, and `CHICAGOCHICAGO`.

We first correct obvious formatting and typing issues, then use the cleaned ZIP codes to impute missing City values.  
Imputation is performed only when historical ZIP-to-City consistency is **≥ 90%**.


In [ ]:
# Basic standardization
food_test["City Clean"] = (
    food_test["City"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\.$", "", regex=True)
)

# Only corrections that were manually reviewed as clear errors
city_corrections = {
    "CCHICAGO": "CHICAGO",
    "CHICAGOO": "CHICAGO",
    "CHICAGOCHICAGO": "CHICAGO",
    "312CHICAGO": "CHICAGO",
    "CHCHICAGO": "CHICAGO",
    "CHICAGOI": "CHICAGO",
    "CHICAGOC": "CHICAGO",
    "CHCICAGO": "CHICAGO",
    "INACTIVE": "CHICAGO",
    "CHICAGOBEDFORD PARK": "CHICAGO",
    "CH": "CHICAGO",
    "CHARLES A HAYES": "CHICAGO",
    "MERRIVILLE": "MERRILLVILLE",
    "OOLYMPIA FIELDS": "OLYMPIA FIELDS",
    "NILES NILES": "NILES"
}

food_test["City Clean"] = food_test["City Clean"].replace(city_corrections)

# ZIP -> City lookup after City cleaning
city_zip_stats = (
    food_test.dropna(subset=["Zip Imputed", "City Clean"])
    .groupby("Zip Imputed")["City Clean"]
    .agg(
        predicted=lambda x: x.mode().iloc[0],
        total_records="count",
        mode_count=lambda x: x.value_counts().iloc[0]
    )
)
city_zip_stats["confidence"] = (
    city_zip_stats["mode_count"] / city_zip_stats["total_records"]
)

food_test["City Final"] = food_test["City Clean"]
food_test["City Imputation Method"] = np.where(
    food_test["City Clean"].notna(),
    "Original",
    "Still Missing"
)

prediction = food_test["Zip Imputed"].map(city_zip_stats["predicted"])
confidence = food_test["Zip Imputed"].map(city_zip_stats["confidence"])

mask = (
    food_test["City Final"].isna()
    & prediction.notna()
    & (confidence >= 0.90)
)

food_test.loc[mask, "City Final"] = prediction[mask]
food_test.loc[mask, "City Imputation Method"] = "ZIP"

print(food_test["City Imputation Method"].value_counts())
print("Remaining missing City:", food_test["City Final"].isna().sum())

## 6. State

All 75 records with missing State values have `CHICAGO` as their cleaned City.  
Therefore, we fill missing State values with `IL` only when `City Final == "CHICAGO"`.

Existing values of IN / CA / WI / DC / CO / NY are preserved because their City and ZIP values are consistent with those states. They should not be automatically changed to IL.


In [ ]:
food_test["State Final"] = food_test["State"]
food_test["State Imputation Method"] = np.where(
    food_test["State"].notna(),
    "Original",
    "Still Missing"
)

mask = (
    food_test["State Final"].isna()
    & (food_test["City Final"] == "CHICAGO")
)

food_test.loc[mask, "State Final"] = "IL"
food_test.loc[mask, "State Imputation Method"] = "City"

print(food_test["State Imputation Method"].value_counts())
print("Remaining missing State:", food_test["State Final"].isna().sum())

## 7. Coordinates, Results, Inspection Type, Violations

- **Latitude / Longitude**: Missing coordinates have no reliable exact-address reference, so they remain NaN. Using a ZIP centroid would create a misleading impression of precise location, so we do not impute coordinates.
- **Results**: There are no missing values. All seven result categories have meaningful interpretations, so their original values are preserved.
- **Inspection Type**: Only one record is missing this value, so that row is removed. The remaining values receive whitespace normalization only.
- **Violations**: NaN does not necessarily indicate lost data. Records with outcomes such as `Pass`, `Out of Business`, or `No Entry` may reasonably have no violation text. Therefore, we neither impute missing Violations nor remove rows solely because Violations is missing.


In [ ]:
# Remove the single row with missing Inspection Type
food_test = food_test.dropna(subset=["Inspection Type"]).copy()

food_test["Inspection Type Clean"] = (
    food_test["Inspection Type"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Keep a flag for spatial analyses; do not impute fake coordinates
food_test["Coordinates Missing"] = (
    food_test["Latitude"].isna()
    | food_test["Longitude"].isna()
)

print("Results missing:", food_test["Results"].isna().sum())
print("Violations missing:", food_test["Violations"].isna().sum())
print("Coordinate pairs missing:", food_test["Coordinates Missing"].sum())

## 8. Build Final Food Dataset

Keep only the final columns needed for analysis, excluding helper columns used during cleaning.


In [ ]:
food_clean = food_test[
    [
        "Inspection ID",
        "DBA Name",
        "AKA Name",
        "License #",
        "Facility Type Final",
        "Risk Final",
        "Address",
        "City Final",
        "State Final",
        "Zip Imputed",
        "Inspection Date",
        "Inspection Type Clean",
        "Results",
        "Violations",
        "Latitude",
        "Longitude",
        "Location",
        "Year"
    ]
].copy()

food_clean = food_clean.rename(columns={
    "Facility Type Final": "Facility Type",
    "Risk Final": "Risk",
    "City Final": "City",
    "State Final": "State",
    "Zip Imputed": "Zip",
    "Inspection Type Clean": "Inspection Type"
})

# Final data types
food_clean["Inspection Date"] = pd.to_datetime(
    food_clean["Inspection Date"],
    format="mixed",
    errors="coerce"
)

food_clean["Zip"] = (
    pd.to_numeric(food_clean["Zip"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

food_clean["License #"] = (
    pd.to_numeric(food_clean["License #"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

text_cols = [
    "DBA Name", "AKA Name", "Facility Type", "Risk",
    "Address", "City", "State", "Inspection Type",
    "Results", "Violations"
]

for col in text_cols:
    food_clean[col] = food_clean[col].astype("string")

print("Food final shape:", food_clean.shape)
print("\nMissing values:")
print(food_clean.isna().sum().sort_values(ascending=False))
print("\nDuplicate Inspection IDs:", food_clean["Inspection ID"].duplicated().sum())

In [ ]:
food_clean.to_csv(
    "/Users/michael818/Desktop/BA780_dataCleaning/Food_Inspections_Cleaned.csv",
    index=False
)

# Part B — 311 Sanitation

This dataset already combines Historical and Current records.  
The main analysis uses **ZIP Code + Year** to calculate sanitation complaint counts, so cleaning focuses on:

1. Correctly parsing the mixed date formats from Historical and Current records.
2. Removing exact duplicate rows.
3. Prioritizing recovery of missing ZIP codes.
4. Avoiding unnecessary imputation of geographic or detail fields solely for completeness.


## 9. Parse Dates Correctly

Historical and Current 311 records use different date string formats, so `format="mixed"` is required.  
Using a standard `pd.to_datetime(..., errors="coerce")` call without this setting could incorrectly convert many Current dates to `NaT`.


In [ ]:
sanitation_test["Created Date"] = pd.to_datetime(
    sanitation_test["Created Date"],
    format="mixed",
    errors="coerce"
)

sanitation_test["Closed Date"] = pd.to_datetime(
    sanitation_test["Closed Date"],
    format="mixed",
    errors="coerce"
)

print("Missing Created Date:", sanitation_test["Created Date"].isna().sum())
print(
    "Date range:",
    sanitation_test["Created Date"].min(),
    "to",
    sanitation_test["Created Date"].max()
)

## 10. Remove Exact Duplicates

Remove rows only when **all 13 columns are identical**.  
Do not deduplicate using `Request Number` alone because records with the same Request Number may contain different information.


In [ ]:
before = len(sanitation_test)
sanitation_test = sanitation_test.drop_duplicates().copy()

print("Exact duplicates removed:", before - len(sanitation_test))
print("Remaining exact duplicates:", sanitation_test.duplicated().sum())

## 11. Impute Missing ZIP from Coordinates

ZIP is required for the later `ZIP + Year` aggregation.

Many missing ZIP values occur in more recent years, so deleting those rows would systematically exclude recent records.  
Most of these records contain Latitude / Longitude, allowing ZIP imputation using the nearest geographic neighbor.

Previous holdout validation using 50,000 records with known ZIP codes showed:
- Accuracy: **99.31%**
- Mean nearest distance: approximately **0.01 km**

We therefore retain this method. Only the small number of rows still missing ZIP and lacking usable geographic information are removed.


In [ ]:
# Known ZIP + coordinate records become the geographic reference
reference = sanitation_test[
    sanitation_test["ZIP Code"].notna()
    & sanitation_test["Latitude"].notna()
    & sanitation_test["Longitude"].notna()
].copy()

ref_coords = np.radians(
    reference[["Latitude", "Longitude"]].to_numpy()
)

tree = BallTree(ref_coords, metric="haversine")

# Only impute rows where ZIP is missing but coordinates are available
missing_zip_mask = (
    sanitation_test["ZIP Code"].isna()
    & sanitation_test["Latitude"].notna()
    & sanitation_test["Longitude"].notna()
)

missing_coords = np.radians(
    sanitation_test.loc[
        missing_zip_mask,
        ["Latitude", "Longitude"]
    ].to_numpy()
)

if len(missing_coords) > 0:
    distances, indices = tree.query(missing_coords, k=1)

    sanitation_test.loc[
        missing_zip_mask,
        "ZIP Code"
    ] = reference.iloc[
        indices.flatten()
    ]["ZIP Code"].to_numpy()

print("ZIPs imputed:", missing_zip_mask.sum())
print("ZIPs still missing:", sanitation_test["ZIP Code"].isna().sum())

# Remaining rows have no reliable geographic information for the ZIP+Year analysis
before = len(sanitation_test)

sanitation_test = sanitation_test[
    sanitation_test["ZIP Code"].notna()
].copy()

print("Rows removed because ZIP could not be recovered:", before - len(sanitation_test))

## 12. Final 311 Data Types

- ZIP is an identifier, so store it as string rather than float.
- Ward / Police District / Community Area are integer-like identifiers and may remain nullable.
- `Closed Date` missing values are retained because open requests may legitimately have no close date.
- Small remaining missing values in geographic detail fields are retained because the main analysis only requires ZIP + Year.

In [ ]:
sanitation_test["ZIP Code"] = (
    pd.to_numeric(sanitation_test["ZIP Code"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

for col in ["Ward", "Police District", "Community Area"]:
    sanitation_test[col] = (
        pd.to_numeric(sanitation_test[col], errors="coerce")
        .astype("Int64")
    )

for col in ["Latitude", "Longitude"]:
    sanitation_test[col] = pd.to_numeric(
        sanitation_test[col],
        errors="coerce"
    )

sanitation_test["Year"] = sanitation_test["Year"].astype("Int64")

print("311 final shape:", sanitation_test.shape)
print("\nMissing values:")
print(sanitation_test.isna().sum().sort_values(ascending=False))
print("\nDate range:")
print(
    sanitation_test["Created Date"].min(),
    "to",
    sanitation_test["Created Date"].max()
)
print("\nExact duplicates:", sanitation_test.duplicated().sum())

In [ ]:
sanitation_test.to_csv(
    "/Users/michael818/Desktop/BA780_dataCleaning/311_Sanitation_Cleaned_2010_2026.csv",
    index=False
)

# Final Notes for Analysis

Both cleaned datasets are now ready for subsequent analysis.

### Food Inspections
- Do not delete entire rows solely because `Violations`, `Facility Type`, or coordinates contain NaN.
- When analyzing a specific field, drop missing values only within the relevant analysis subset.

### 311 Sanitation
- The year 2010 contains very few records, so annual coverage is clearly incomplete.
- The year 2026 is a partial year, with data available only through September.
- For yearly trends, do not compare 2010 or 2026 directly with full years as if their coverage were complete.

### Next Step
Aggregate 311 records into:

`ZIP Code + Year → Sanitation Complaint Count`

Then merge with Food Inspections using `Zip + Year`.
